# EnergiAI — EDA, Modelado y Serialización

**Frente:** EDA + comparación de modelos + serialización + notebook (Ciencia de Datos)

**Objetivo de este notebook:**
1. Explorar y limpiar el dataset de entrenamiento (`dataset_entrenamiento.csv`)
2. Analizar patrones de consumo energético
3. Procesar y transformar variables para el modelado
4. Entrenar y comparar tres modelos supervisados: Regresión Logística, Árbol de Decisión y Random Forest
5. Evaluar con métricas adecuadas (accuracy, F1 por clase, matriz de confusión)
6. Generar recomendaciones basadas en reglas
7. Serializar el mejor modelo con Joblib

**Entrada:** `salidas_energiai/dataset_entrenamiento.csv`, generado por `Energia.py` → `etiquetar_dataset.py`.

**Nota importante sobre la variable objetivo:** la columna `categoria` fue generada por
`etiquetar_dataset.py` con un sistema de puntos basado en percentiles del propio dataset
(no son valores fijos). Como la etiqueta se deriva de reglas propias, es esperable que el
modelo re-aprenda esas reglas con métricas muy altas. El valor de este ejercicio está en
comparar arquitecturas y en documentar el criterio de etiquetado, no en perseguir un
accuracy "sorpresivo".


## 1. Configuración e imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
RUTA_DATASET = Path("salidas_energiai/dataset_entrenamiento.csv")
RUTA_MODELOS = Path("modelos")
RUTA_MODELOS.mkdir(exist_ok=True)


## 2. Carga de datos

Cargamos el dataset de entrenamiento que produce `etiquetar_dataset.py`. Debe tener
exactamente 6 columnas: las 5 del contrato de la API + `categoria`.


In [ ]:
df = pd.read_csv(RUTA_DATASET)

print(f"Filas: {len(df)}  |  Columnas: {list(df.columns)}")
df.head()


## 3. Exploración y limpieza de datos (EDA)

### 3.1 Estructura general y tipos de datos

In [ ]:
df.info()


### 3.2 Valores nulos

El dataset no debería traer nulos: `Energia.py` ya imputa por mediana los valores
faltantes durante el procesamiento de ENCEVI. Lo verificamos de todas formas.


In [ ]:
nulos = df.isna().sum()
print("Valores nulos por columna:")
print(nulos)

if nulos.sum() == 0:
    print("\nSin valores nulos. No se requiere imputación adicional.")


### 3.3 Duplicados

Verificamos que no haya filas exactamente duplicadas (hogares repetidos por error de
fusión, por ejemplo).


In [ ]:
duplicados = df.duplicated().sum()
print(f"Filas duplicadas: {duplicados} ({duplicados / len(df) * 100:.2f}%)")


### 3.4 Estadísticos descriptivos de las variables numéricas


In [ ]:
df.describe().T


### 3.5 Distribución de la variable objetivo (`categoria`)

Balance de clases: importante para decidir si conviene usar `class_weight='balanced'`
o alguna técnica de remuestreo. Los umbrales de `etiquetar_dataset.py` buscan que
ninguna clase caiga por debajo del 15%.


In [ ]:
conteo_categoria = df["categoria"].value_counts()
porcentaje_categoria = (conteo_categoria / len(df) * 100).round(1)

resumen_categoria = pd.DataFrame({
    "conteo": conteo_categoria,
    "porcentaje": porcentaje_categoria,
})
print(resumen_categoria)

fig, ax = plt.subplots(figsize=(6, 4))
orden = ["EFICIENTE", "MODERADO", "INEFICIENTE"]
sns.countplot(data=df, x="categoria", order=orden, palette="viridis", ax=ax)
ax.set_title("Distribución de la variable objetivo: categoría energética")
ax.set_xlabel("Categoría")
ax.set_ylabel("Cantidad de hogares")
plt.tight_layout()
plt.show()


### 3.6 Distribución de las variables numéricas

Miramos `consumo_kwh`, `cantidad_equipos` y `horas_alto_consumo`. Se espera una
distribución sesgada a la derecha (cola larga de hogares de alto consumo), típica de
datos de energía residencial.


In [ ]:
variables_numericas = ["consumo_kwh", "cantidad_equipos", "horas_alto_consumo"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, variables_numericas):
    sns.histplot(df[col], bins=40, kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribución de {col}")
plt.tight_layout()
plt.show()


### 3.7 Variables categóricas y booleanas


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=df, x="tipo_inmueble", order=df["tipo_inmueble"].value_counts().index,
              palette="crest", ax=axes[0])
axes[0].set_title("Distribución de tipo_inmueble")

sns.countplot(data=df, x="uso_horario_pico", palette="crest", ax=axes[1])
axes[1].set_title("Distribución de uso_horario_pico")

plt.tight_layout()
plt.show()

print(df["tipo_inmueble"].value_counts(normalize=True).round(3) * 100)
print()
print(df["uso_horario_pico"].value_counts(normalize=True).round(3) * 100)


## 4. Análisis de patrones de consumo

Cruzamos las variables de entrada contra la categoría para entender qué combina cada
perfil de eficiencia. Esto también sirve como evidencia de que las 5 variables
realmente están relacionadas con la etiqueta, no solo `consumo_kwh`.


### 4.1 Consumo (kWh) por categoría

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="categoria", y="consumo_kwh", order=orden, palette="viridis", ax=ax)
ax.set_title("Consumo mensual (kWh) por categoría")
plt.tight_layout()
plt.show()

df.groupby("categoria")["consumo_kwh"].describe().loc[orden]


### 4.2 Horas de alto consumo por categoría

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="categoria", y="horas_alto_consumo", order=orden, palette="viridis", ax=ax)
ax.set_ylim(0, df["horas_alto_consumo"].quantile(0.95))  # recortamos outliers extremos para visualizar mejor
ax.set_title("Horas de alto consumo por categoría (recortado al percentil 95)")
plt.tight_layout()
plt.show()


### 4.3 Cantidad de equipos por categoría

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="categoria", y="cantidad_equipos", order=orden, palette="viridis", ax=ax)
ax.set_title("Cantidad de equipos por categoría")
plt.tight_layout()
plt.show()


### 4.4 Uso en horario pico por categoría (tabla cruzada)


In [ ]:
tabla_pico = pd.crosstab(df["categoria"], df["uso_horario_pico"], normalize="index").round(3) * 100
tabla_pico.loc[orden]


### 4.5 Intensidad de consumo por equipo (consumo_kwh / cantidad_equipos)

Esta es la variable derivada que usa `etiquetar_dataset.py` como cuarto factor de
puntaje: no penaliza tener muchos equipos, penaliza consumir mucho *por equipo*.


In [ ]:
df["intensidad_kwh_por_equipo"] = df["consumo_kwh"] / df["cantidad_equipos"].clip(lower=1)

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=df, x="categoria", y="intensidad_kwh_por_equipo", order=orden, palette="viridis", ax=ax)
ax.set_ylim(0, df["intensidad_kwh_por_equipo"].quantile(0.95))
ax.set_title("Intensidad de consumo por equipo, por categoría (recortado al percentil 95)")
plt.tight_layout()
plt.show()


### 4.6 Matriz de correlación (variables numéricas)


In [ ]:
cols_correlacion = ["consumo_kwh", "cantidad_equipos", "horas_alto_consumo", "intensidad_kwh_por_equipo"]
matriz_corr = df[cols_correlacion].corr()

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(matriz_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlación entre variables numéricas")
plt.tight_layout()
plt.show()


**Conclusión del EDA:** las cuatro variables (consumo, horas de alto consumo,
cantidad de equipos, uso en horario pico) muestran diferencias claras entre las tres
categorías, y no están perfectamente correlacionadas entre sí — lo cual respalda que
el sistema de etiquetado multifactor efectivamente combina información distinta de
cada una, en vez de depender de una sola señal dominante.


## 5. Procesamiento y transformación de variables

Preparamos las variables para el modelado:
- **Numéricas** (`consumo_kwh`, `cantidad_equipos`, `horas_alto_consumo`): se
  estandarizan con `StandardScaler` (necesario para Regresión Logística; no afecta a
  los modelos basados en árboles, así que lo dejamos igual para las tres pipelines por
  simplicidad y consistencia).
- **Categórica** (`tipo_inmueble`): se codifica con `OneHotEncoder`.
- **Booleana** (`uso_horario_pico`): se deja pasar tal cual (ya es 0/1 implícito).

Usamos `ColumnTransformer` + `Pipeline` de scikit-learn para que el preprocesamiento
quede empaquetado junto con el modelo al serializar — así el backend/inference-service
no tiene que reimplementar la lógica de codificación por su cuenta.


In [ ]:
COLUMNAS_ENTRADA = ["consumo_kwh", "uso_horario_pico", "cantidad_equipos",
                     "tipo_inmueble", "horas_alto_consumo"]

X = df[COLUMNAS_ENTRADA].copy()
y = df["categoria"].copy()

columnas_numericas = ["consumo_kwh", "cantidad_equipos", "horas_alto_consumo"]
columnas_categoricas = ["tipo_inmueble"]
# uso_horario_pico se deja pasar tal cual vía remainder="passthrough"

preprocesador = ColumnTransformer(
    transformers=[
        ("numericas", StandardScaler(), columnas_numericas),
        ("categoricas", OneHotEncoder(drop="first", handle_unknown="ignore"), columnas_categoricas),
    ],
    remainder="passthrough",
)

print("Variables de entrada:", COLUMNAS_ENTRADA)
print("Variable objetivo: categoria")
print(f"\nX shape: {X.shape}  |  y shape: {y.shape}")


### 5.1 División train / test

Usamos `stratify=y` para que las tres clases mantengan la misma proporción en train y
en test — importante porque las clases no están perfectamente balanceadas.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape[0]} filas  |  Test: {X_test.shape[0]} filas")
print("\nProporción de clases en train:")
print((y_train.value_counts(normalize=True) * 100).round(1))
print("\nProporción de clases en test:")
print((y_test.value_counts(normalize=True) * 100).round(1))


## 6. Entrenamiento de modelos supervisados

Comparamos tres algoritmos, tal como recomienda la descripción del proyecto:

- **Regresión Logística**: modelo lineal, rápido, interpretable, buena referencia base.
- **Árbol de Decisión**: captura relaciones no lineales, fácil de interpretar visualmente.
- **Random Forest**: ensamble de árboles, generalmente más robusto y con mejor
  desempeño que un único árbol.

Cada uno se entrena dentro de un `Pipeline` junto con el `preprocesador`, así el
modelo final serializado incluye tanto el preprocesamiento como la clasificación.


In [ ]:
modelos_candidatos = {
    "Regresion Logistica": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_STATE,
    ),
    "Arbol de Decision": DecisionTreeClassifier(
        max_depth=6,
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

pipelines_entrenados = {}

for nombre, modelo in modelos_candidatos.items():
    pipeline = Pipeline(steps=[
        ("preprocesador", preprocesador),
        ("clasificador", modelo),
    ])
    pipeline.fit(X_train, y_train)
    pipelines_entrenados[nombre] = pipeline
    print(f"{nombre}: entrenado.")


## 7. Evaluación con métricas adecuadas

Para cada modelo calculamos:
- **Accuracy**: porcentaje global de aciertos.
- **F1-macro**: promedio del F1 de cada clase sin ponderar por tamaño — penaliza si el
  modelo ignora la clase minoritaria (EFICIENTE, ~24% del dataset).
- **Matriz de confusión**: para ver en qué categorías se equivoca cada modelo.
- **Validación cruzada (5-fold)** sobre el set de entrenamiento, para tener una
  estimación más robusta que un único split train/test.


In [ ]:
resultados = []

for nombre, pipeline in pipelines_entrenados.items():
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")

    cv_scores = cross_val_score(
        pipeline, X_train, y_train, cv=5, scoring="accuracy", n_jobs=-1
    )

    resultados.append({
        "modelo": nombre,
        "accuracy_test": acc,
        "f1_macro_test": f1_macro,
        "accuracy_cv_mean": cv_scores.mean(),
        "accuracy_cv_std": cv_scores.std(),
    })

    print(f"\n{'=' * 60}")
    print(f"{nombre}")
    print(f"{'=' * 60}")
    print(f"Accuracy (test):      {acc:.4f}")
    print(f"F1-macro (test):      {f1_macro:.4f}")
    print(f"Accuracy (5-fold CV): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
    print("\nReporte por clase:")
    print(classification_report(y_test, y_pred, target_names=orden))

tabla_resultados = pd.DataFrame(resultados).sort_values("f1_macro_test", ascending=False)
tabla_resultados


### 7.1 Matrices de confusión

Visualizamos la matriz de confusión de cada modelo para comparar en qué categorías
se equivocan más.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (nombre, pipeline) in zip(axes, pipelines_entrenados.items()):
    y_pred = pipeline.predict(X_test)
    cm = confusion_matrix(y_test, y_pred, labels=orden)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=orden)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
    ax.set_title(nombre)

plt.tight_layout()
plt.show()


### 7.2 Comparación visual de métricas


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
tabla_resultados.set_index("modelo")[["accuracy_test", "f1_macro_test"]].plot(
    kind="bar", ax=ax, color=["steelblue", "coral"]
)
ax.set_ylabel("Score")
ax.set_title("Comparación de modelos: Accuracy vs F1-macro")
ax.set_ylim(0, 1.05)
ax.legend(["Accuracy (test)", "F1-macro (test)"])
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### 7.3 Selección del mejor modelo

Elegimos el modelo con mejor **F1-macro en test**, porque es la métrica que mejor
refleja el desempeño equilibrado entre las tres clases (no solo la accuracy global,
que puede estar dominada por la clase mayoritaria INEFICIENTE).


In [ ]:
nombre_mejor_modelo = tabla_resultados.iloc[0]["modelo"]
mejor_pipeline = pipelines_entrenados[nombre_mejor_modelo]

print(f"Mejor modelo seleccionado: {nombre_mejor_modelo}")
print(f"F1-macro (test): {tabla_resultados.iloc[0]['f1_macro_test']:.4f}")
print(f"Accuracy (test): {tabla_resultados.iloc[0]['accuracy_test']:.4f}")


### 7.4 Importancia de variables (solo para modelos basados en árboles)

Si el mejor modelo es Árbol de Decisión o Random Forest, revisamos qué tanto pesa
cada variable en la decisión final. Esto es clave para el proyecto: confirma que las
5 variables del contrato realmente aportan a la clasificación (y no solo
`consumo_kwh`, como pasaba con el etiquetado original por terciles).


In [ ]:
if hasattr(mejor_pipeline.named_steps["clasificador"], "feature_importances_"):
    nombres_columnas_transformadas = mejor_pipeline.named_steps["preprocesador"].get_feature_names_out()
    importancias = mejor_pipeline.named_steps["clasificador"].feature_importances_

    df_importancias = pd.DataFrame({
        "variable": nombres_columnas_transformadas,
        "importancia": importancias,
    }).sort_values("importancia", ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.barplot(data=df_importancias, x="importancia", y="variable", palette="viridis", ax=ax)
    ax.set_title(f"Importancia de variables — {nombre_mejor_modelo}")
    plt.tight_layout()
    plt.show()

    df_importancias
else:
    print(f"{nombre_mejor_modelo} no expone feature_importances_ directamente "
          f"(es un modelo lineal). Se omite este análisis.")


### 7.5 Matriz de confusión normalizada (en %)

La matriz de confusión anterior muestra conteos absolutos. Aquí la normalizamos por
fila para leerla como porcentajes: de todos los hogares que realmente son X, qué
porcentaje el modelo predijo en cada categoría.

In [ ]:
cm_normalizada = confusion_matrix(y_test, mejor_pipeline.predict(X_test), labels=orden, normalize="true")

fig, ax = plt.subplots(figsize=(6, 5))
disp_norm = ConfusionMatrixDisplay(confusion_matrix=cm_normalizada, display_labels=orden)
disp_norm.plot(ax=ax, cmap="Blues", colorbar=True, values_format=".1%", xticks_rotation=45)
ax.set_title(f"Matriz de confusión normalizada — {nombre_mejor_modelo}")
plt.tight_layout()
plt.show()

### 7.6 Curva de aprendizaje del mejor modelo

Con un accuracy tan alto, verificamos que el modelo no esté sobreajustando: comparamos
el desempeño en entrenamiento vs. validación cruzada a medida que crece el dataset.
Si no hay sobreajuste severo, ambas curvas deben converger sin dejar una brecha grande.

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    mejor_pipeline,
    X_train, y_train,
    cv=5,
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 8),
    scoring="accuracy",
    random_state=RANDOM_STATE,
)

train_media = train_scores.mean(axis=1)
val_media = val_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(train_sizes, train_media, marker="o", label="Score en entrenamiento")
ax.plot(train_sizes, val_media, marker="o", label="Score en validación cruzada")
ax.fill_between(train_sizes, train_scores.min(axis=1), train_scores.max(axis=1), alpha=0.15)
ax.fill_between(train_sizes, val_scores.min(axis=1), val_scores.max(axis=1), alpha=0.15)
ax.set_xlabel("Cantidad de ejemplos de entrenamiento")
ax.set_ylabel("Accuracy")
ax.set_title(f"Curva de aprendizaje — {nombre_mejor_modelo}")
ax.legend(loc="lower right")
ax.set_ylim(0.9, 1.01)
plt.tight_layout()
plt.show()

brecha_final = train_media[-1] - val_media[-1]
print(f"Brecha train-validación con el dataset completo: {brecha_final:.4f}")
if brecha_final < 0.02:
    print("Brecha pequeña: no hay evidencia fuerte de sobreajuste.")
else:
    print("Brecha considerable: podría valer la pena regularizar más el modelo.")

## 8. Generación de recomendaciones basadas en reglas

Además de clasificar, la API debe devolver recomendaciones textuales para ayudar al
usuario a reducir su consumo. Usamos reglas simples e interpretables (no un modelo)
porque el objetivo aquí es dar consejos accionables y explicables, no predecir algo.

Las reglas están alineadas con los mismos factores que usa `etiquetar_dataset.py`
para puntuar el perfil energético — así las recomendaciones son consistentes con el
criterio de clasificación.


In [ ]:
def generar_recomendaciones(consumo_kwh, uso_horario_pico, cantidad_equipos,
                             tipo_inmueble, horas_alto_consumo, categoria):
    """
    Genera una lista de recomendaciones de texto según el perfil de consumo.

    Esta función replica la lógica que usará el backend (o el inference-service)
    después de recibir la categoría del modelo: el modelo solo clasifica, el resto
    del pipeline (costo + recomendaciones) es lógica de negocio basada en reglas.
    """
    recomendaciones = []

    if uso_horario_pico:
        recomendaciones.append(
            "Reducir el uso de equipos durante los horarios pico"
        )

    intensidad = consumo_kwh / max(cantidad_equipos, 1)
    if intensidad > 15:
        recomendaciones.append(
            "Evaluar equipos con alto consumo energético"
        )

    if horas_alto_consumo > 2:
        recomendaciones.append(
            "Distribuir las actividades de mayor consumo a lo largo del día"
        )

    if categoria == "INEFICIENTE" and cantidad_equipos > 15:
        recomendaciones.append(
            "Considerar sustituir los electrodomésticos más antiguos por modelos "
            "de mayor eficiencia energética"
        )

    if categoria == "EFICIENTE" and not recomendaciones:
        recomendaciones.append(
            "Buen perfil energético: mantener los hábitos actuales de consumo"
        )

    if not recomendaciones:
        recomendaciones.append(
            "Continuar monitoreando el consumo mensual para mantener la eficiencia"
        )

    return recomendaciones


### 8.1 Estimación del costo mensual

Usando la tarifa de referencia acordada en el proyecto: **$0.75 por kWh**.


In [ ]:
TARIFA_REFERENCIA_KWH = 0.75

def estimar_costo_mensual(consumo_kwh, tarifa=TARIFA_REFERENCIA_KWH):
    return round(consumo_kwh * tarifa, 2)


## 9. Ejemplos de uso end-to-end (mínimo 3, según el contrato)

Simulamos tres solicitudes tal como llegarían desde el backend, siguiendo exactamente
el contrato JSON de la API, y mostramos la salida completa: categoría, probabilidad,
costo estimado y recomendaciones.


In [ ]:
def analizar_consumo(consumo_kwh, uso_horario_pico, cantidad_equipos,
                      tipo_inmueble, horas_alto_consumo, pipeline=mejor_pipeline):
    """Replica el endpoint POST /analisis-energetico usando el modelo entrenado."""
    entrada = pd.DataFrame([{
        "consumo_kwh": consumo_kwh,
        "uso_horario_pico": uso_horario_pico,
        "cantidad_equipos": cantidad_equipos,
        "tipo_inmueble": tipo_inmueble,
        "horas_alto_consumo": horas_alto_consumo,
    }])

    categoria_predicha = pipeline.predict(entrada)[0]
    probabilidades = pipeline.predict_proba(entrada)[0]
    clases = pipeline.named_steps["clasificador"].classes_
    probabilidad = float(max(probabilidades))

    recomendaciones = generar_recomendaciones(
        consumo_kwh, uso_horario_pico, cantidad_equipos,
        tipo_inmueble, horas_alto_consumo, categoria_predicha,
    )

    costo_estimado = estimar_costo_mensual(consumo_kwh)

    return {
        "categoria": categoria_predicha,
        "probabilidad": round(probabilidad, 2),
        "costo_estimado_mensual": costo_estimado,
        "tarifa_referencia_kwh": TARIFA_REFERENCIA_KWH,
        "recomendaciones": recomendaciones,
    }


In [ ]:
# Ejemplo 1: el mismo ejemplo del contrato de la API en la documentación del proyecto
ejemplo_1 = analizar_consumo(
    consumo_kwh=420,
    uso_horario_pico=True,
    cantidad_equipos=10,
    tipo_inmueble="Casa",
    horas_alto_consumo=8,
)

print("Ejemplo 1 (hogar de alto consumo, del contrato de la API):")
import json
print(json.dumps(ejemplo_1, indent=2, ensure_ascii=False))


In [ ]:
# Ejemplo 2: hogar de consumo bajo / probablemente eficiente
ejemplo_2 = analizar_consumo(
    consumo_kwh=45,
    uso_horario_pico=False,
    cantidad_equipos=4,
    tipo_inmueble="Departamento",
    horas_alto_consumo=0,
)

print("Ejemplo 2 (hogar de bajo consumo):")
print(json.dumps(ejemplo_2, indent=2, ensure_ascii=False))


In [ ]:
# Ejemplo 3: hogar de consumo intermedio
ejemplo_3 = analizar_consumo(
    consumo_kwh=150,
    uso_horario_pico=True,
    cantidad_equipos=8,
    tipo_inmueble="Casa",
    horas_alto_consumo=1.5,
)

print("Ejemplo 3 (hogar de consumo intermedio):")
print(json.dumps(ejemplo_3, indent=2, ensure_ascii=False))


## 10. Serialización del modelo entrenado

Guardamos el pipeline completo (preprocesamiento + clasificador) con Joblib, listo
para subir a OCI Object Storage y para que el `inference-service` en Python lo cargue
y lo use directamente con `pipeline.predict(entrada)`.


In [ ]:
ruta_modelo = RUTA_MODELOS / "modelo_energiai.joblib"

joblib.dump(mejor_pipeline, ruta_modelo)

print(f"Modelo serializado en: {ruta_modelo}")
print(f"Modelo elegido: {nombre_mejor_modelo}")
print(f"Tamaño del archivo: {ruta_modelo.stat().st_size / 1024:.1f} KB")


### 10.1 Verificación: recargar el modelo y probar que funciona igual

Antes de entregarlo al equipo de Backend, confirmamos que el archivo `.joblib` se
puede cargar de nuevo desde disco y produce exactamente el mismo resultado.


In [ ]:
modelo_recargado = joblib.load(ruta_modelo)

prueba_recarga = modelo_recargado.predict(pd.DataFrame([{
    "consumo_kwh": 420,
    "uso_horario_pico": True,
    "cantidad_equipos": 10,
    "tipo_inmueble": "Casa",
    "horas_alto_consumo": 8,
}]))

print("Predicción tras recargar el modelo desde disco:", prueba_recarga[0])
assert prueba_recarga[0] == ejemplo_1["categoria"], "La predicción cambió al recargar el modelo"
print("Verificación correcta: el modelo recargado predice lo mismo que el original.")


### 10.2 Reporte de métricas en JSON

Guardamos un archivo `reporte_metricas.json` junto al modelo, con el resumen de
métricas y metadatos, para que Backend u OCI lo consulten sin abrir el notebook.

In [ ]:
fila_mejor_modelo = tabla_resultados[tabla_resultados["modelo"] == nombre_mejor_modelo].iloc[0]

reporte_metricas = {
    "modelo_elegido": nombre_mejor_modelo,
    "fecha_entrenamiento": pd.Timestamp.now().strftime("%Y-%m-%dT%H:%M:%SZ"),
    "metricas_test": {
        "accuracy": round(float(fila_mejor_modelo["accuracy_test"]), 4),
        "f1_macro": round(float(fila_mejor_modelo["f1_macro_test"]), 4),
    },
    "metricas_validacion_cruzada_5fold": {
        "accuracy_mean": round(float(fila_mejor_modelo["accuracy_cv_mean"]), 4),
        "accuracy_std": round(float(fila_mejor_modelo["accuracy_cv_std"]), 4),
    },
    "variables_entrada": COLUMNAS_ENTRADA,
    "clases": orden,
    "tarifa_referencia_kwh": TARIFA_REFERENCIA_KWH,
    "filas_entrenamiento": int(len(X_train)),
    "filas_prueba": int(len(X_test)),
    "archivo_modelo": str(ruta_modelo.name),
}

ruta_reporte = RUTA_MODELOS / "reporte_metricas.json"
with open(ruta_reporte, "w", encoding="utf-8") as f:
    json.dump(reporte_metricas, f, indent=2, ensure_ascii=False)

print(f"Reporte guardado en: {ruta_reporte}")
print(json.dumps(reporte_metricas, indent=2, ensure_ascii=False))

## 11. Resumen
- **Dataset**: 28,764+ hogares reales derivados de ENCEVI 2018, con 5 variables de
  entrada + `categoria` etiquetada por sistema de puntos multifactor (percentiles).
- **Mejor modelo**: seleccionado automáticamente arriba según F1-macro en test.
- **Modelo serializado**: `modelos/modelo_energiai.joblib`, listo para subir a OCI
  Object Storage.


**Limitación declarada**: `consumo_kwh` en el dataset de entrenamiento es una
estimación física derivada de los aparatos declarados en ENCEVI (potencia × horas de
uso × cantidad), no una lectura de medidor. En producción, este valor lo reporta el
propio usuario desde su recibo de luz.
